# 第18周 · 强化学习：老虎机、Q-learning、REINFORCE 与奖励欺骗

**本周结束时你能做到**：说清智能体、状态、动作、奖励、回报与折扣；在多臂老虎机上实现 ε-greedy 与 UCB，用平均奖励、最优动作比例和累计遗憾比较"探索与利用"；把 Bellman 方程读成"今天的价值 = 即时奖励 + 明天的价值"，从零写出表格型 Q-learning 并在 FrozenLake 上训练；从零实现策略梯度 REINFORCE，亲眼看到它在不同种子之间的巨大方差；用 MountainCar 演示稀疏奖励学不会、势函数塑形学得会、写错的奖励被智能体"钻空子"；对 PPO 与 RLHF 有概念认识。

前 17 周的模型都在**拟合一份固定的数据**。强化学习不一样：数据是智能体自己**一边行动一边收集**的——它做什么决定了它能看到什么。这带来两个新问题，贯穿今天四节课：**探索**（不试就不知道，试了要付代价）和**信用分配**（一局结束才知道输赢，该怪哪一步？）。

本周的蓝本（首次出现处都会再注明）：

- Sutton & Barto, *Reinforcement Learning: An Introduction*, 2nd ed.（2018）第 2 章的 10 臂测试平台——全书 [在作者主页免费提供](http://incompleteideas.net/book/the-book-2nd.html)，许可为 CC BY-NC-ND 2.0，**只链接与引用、不改编**：S1 的代码和图都是课程自己写的；
- [Hugging Face Deep RL Course](https://huggingface.co/learn/deep-rl-course/unit0/introduction) 的 Unit 2（Q-learning）与 Unit 4（REINFORCE），仓库 [huggingface/deep-rl-class](https://github.com/huggingface/deep-rl-class)，Apache-2.0；我们改编了它的函数结构与超参数，并改写成 Gymnasium 1.x 的 API；
- OpenAI Spinning Up 的 [Part 3: Intro to Policy Optimization](https://spinningup.openai.com/en/latest/spinningup/rl_intro3.html)（reward-to-go 与基线的理论，参照阅读）；
- 环境来自 [Gymnasium](https://gymnasium.farama.org/)（Farama Foundation，MIT License）。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")
# 本周新增依赖：Gymnasium（强化学习环境库；只训练、不渲染画面，所以不需要 pygame）
if importlib.util.find_spec("gymnasium") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gymnasium>=1.4,<2"], check=True)
print("Gymnasium 已就绪")

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import gymnasium as gym
from zero2researcher import experiments
from zero2researcher.checks import week18 as ck

# 本周所有 RL 实验固定在 CPU 上跑：网络只有几百个参数、每一步只喂一个状态，搬到 GPU 反而更慢
torch.set_num_threads(1)     # 小网络 + 单个样本时单线程最快，结果也不受线程数影响
print("torch", torch.__version__, "| gymnasium", gym.__version__, "| device: cpu")

---
## S1 · 多臂老虎机：探索与利用

想象一排 10 台老虎机（"臂"）。每台机器的平均回报 $q_*(a)$ 不同，但你不知道；每拉一次只看到一个带噪声的奖励。一共能拉 1000 次——**怎么拉，总奖励最多？**

- 一直拉"目前看起来最好"的那台（**利用**），可能永远错过真正最好的；
- 时不时随便拉一台（**探索**），短期吃亏，但能把估计变准。

这是强化学习最核心的两难，多臂老虎机是它最干净的版本：没有"局面"，这一步拉哪台不影响下一步面对的机器。

本节的实验设定来自 Sutton & Barto 第 2 章的 **10 臂测试平台**（§2.3；下文简称 S&B）：随机生成 2000 个 10 臂任务，每个臂的真值 $q_*(a) \sim \mathcal N(0, 1)$，拉臂 $a$ 得到的奖励 $R \sim \mathcal N(q_*(a), 1)$；每个任务连续拉 1000 次，最后把 2000 条曲线取平均。先看一个任务长什么样：

In [ ]:
rng = np.random.default_rng(0)
q_true = rng.normal(0, 1, size=10)                               # 一个任务：10 个臂的真值（智能体看不到）
pulls = q_true[:, None] + rng.normal(0, 1, size=(10, 200))      # 每个臂各拉 200 次的奖励

fig, ax = plt.subplots(figsize=(7, 3.2))
for arm in range(10):
    ax.scatter(arm + rng.uniform(-0.25, 0.25, 200), pulls[arm], s=3, alpha=0.3, color="C0")
ax.scatter(range(10), q_true, color="C3", marker="_", s=400, lw=3, label="True value q*(a)")
ax.set_xticks(range(10))
ax.set_xlabel("Arm a")
ax.set_ylabel("Reward of a single pull")
ax.set_title(f"Arm {q_true.argmax()} is best, but single pulls of different arms overlap heavily")
ax.legend(loc="upper left")
plt.show()

### 估计每个臂的价值：增量均值

对每个臂只记两个数：拉过的次数 $N(a)$ 和奖励的平均 $Q(a)$。不必存下全部历史——第 $n$ 个奖励 $R_n$ 到来时（S&B 式 2.3），

$$Q_{n+1} = Q_n + \frac{1}{n}\big(R_n - Q_n\big)$$

读作："新估计 = 旧估计 + 步长 × (新奖励 − 旧估计)"。括号里的误差推着估计往真值走，步长 $1/n$ 让每个奖励的权重都相同（就是样本平均）。**"旧值 + 步长 × (目标 − 旧值)"这个形状今天会反复出现**：Q-learning、作业里的滑动平均基线都是它。

### 两种选臂规则

- **ε-greedy**：以概率 $1-\varepsilon$ 选 $\arg\max_a Q(a)$（利用）；以概率 $\varepsilon$ 从全部 10 个臂里均匀随机选一个（探索）。$\varepsilon = 0$ 就是纯贪心。
- **UCB**（upper confidence bound，置信上界，S&B §2.7）：
  $$A_t = \arg\max_a \Big[\, Q(a) + c\,\sqrt{\frac{\ln t}{N(a)}} \,\Big]$$
  第二项是"不确定性加分"：拉得少的臂 $N(a)$ 小、加分多；总步数 $t$ 增长而它一直没被拉，加分又会慢慢涨回来。没拉过的臂（$N(a) = 0$）最优先。探索因此**有方向**，不再是盲目的随机。

下面的函数一次推进 2000 个任务：`Q`、`N` 的形状都是 `(2000, 10)`，`Q[rows, a]` 取出"每个任务被拉的那个臂"。平局时用"加一点点随机噪声再 argmax"的办法随机选（S2 会解释为什么平局必须随机打破）。有一个例外：UCB 里没拉过的臂得分记为 $+\infty$，$\infty$ 加上噪声还是 $\infty$，`np.argmax` 总是选编号最小的那个——所以 UCB 的前 10 步是按编号 0, 1, …, 9 依次试的。S&B 原文是在还没试过的臂里**随机**选；这里每个任务的 10 个臂都是独立随机生成的，先试哪个不影响任何平均结果。

In [ ]:
def run_bandit(epsilon=0.0, ucb_c=None, n_tasks=2000, n_steps=1000, k=10, seed=0):
    # 返回每一步的：平均奖励、选中最优臂的比例、累计（期望）遗憾——都是 n_tasks 个任务的平均
    rng = np.random.default_rng(seed)
    q_star = rng.normal(0.0, 1.0, size=(n_tasks, k))     # 每个任务每个臂的真值
    best = q_star.argmax(axis=1)                          # 每个任务的最优臂
    Q = np.zeros((n_tasks, k))                            # 估计值，初值 0
    N = np.zeros((n_tasks, k))                            # 每个臂被拉过的次数
    rows = np.arange(n_tasks)
    avg_reward, optimal, regret = np.zeros(n_steps), np.zeros(n_steps), np.zeros(n_steps)
    for t in range(n_steps):
        tie_noise = 1e-9 * rng.random((n_tasks, k))       # 加一点点噪声再 argmax = 平局时随机选
        if ucb_c is None:
            # ✍️ 跟着老师一起写
            pass
        else:
            # ✍️ 跟着老师一起写
            pass
        r = rng.normal(q_star[rows, a], 1.0)              # 拉下去，看到带噪声的奖励
        # ✍️ 跟着老师一起写
        pass
        avg_reward[t] = r.mean()
        optimal[t] = (a == best).mean()
        regret[t] = (q_star[rows, best] - q_star[rows, a]).mean()   # 期望遗憾：真值之差（不含奖励噪声）
    return avg_reward, optimal, np.cumsum(regret)

In [ ]:
methods = {
    "Greedy (eps = 0)": dict(epsilon=0.0),
    "eps = 0.01": dict(epsilon=0.01),
    "eps = 0.1": dict(epsilon=0.1),
    "UCB (c = 2)": dict(ucb_c=2.0),
}
t0 = time.time()
bandit = {name: run_bandit(**kw) for name, kw in methods.items()}
print(f"4 种方法 × 2000 个任务 × 1000 步，用时 {time.time() - t0:.1f} 秒")
final_reward = {}
for name, (rew, opt, creg) in bandit.items():
    final_reward[name] = rew[-100:].mean()
    print(f"{name:17s} 最后 100 步平均奖励 {final_reward[name]:.3f} | 最优臂比例 {opt[-100:].mean():5.1%} | "
          f"1000 步累计遗憾 {creg[-1]:6.1f}")
# 与 run_bandit(seed=0) 同一批 2000 个任务：如果每次都拉最优臂，平均奖励是多少
best_possible = np.random.default_rng(0).normal(0, 1, size=(2000, 10)).max(axis=1).mean()
print(f"参照：每次都拉最优臂的平均奖励 = {best_possible:.3f}")
ck.bandit_final_reward(*final_reward.values())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for name, (rew, opt, creg) in bandit.items():
    axes[0].plot(rew, lw=0.8, label=name)
    axes[1].plot(100 * opt, lw=0.8, label=name)
    axes[2].plot(creg, lw=1.5, label=name)
axes[0].axhline(best_possible, color="gray", ls="--", lw=1, label="Always pull the best arm")
axes[0].set(xlabel="Step", ylabel="Average reward (2000 tasks)",
            title="Greedy stalls near 1.0; UCB climbs highest")
axes[1].set(xlabel="Step", ylabel="% of tasks pulling the optimal arm",
            title="Greedy finds the best arm in only about 1/3 of tasks")
axes[2].set(xlabel="Step", ylabel="Cumulative expected regret",
            title="UCB accumulates the least regret")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

怎么读这三张图：

- **贪心**（蓝线）起步不慢，但很快就停在约 1.0：它常常在头几次拉到一台"运气好"的普通机器，从此再也不试别的。只有约三分之一的任务里它锁定了真正最好的臂（S&B 原文也是这么描述的）。
- **ε = 0.1** 探索得多，很快找到最优臂（约 80% 的任务），但它永远要把 10% 的拉杆浪费在随机探索上；**ε = 0.01** 学得慢，但长期看浪费更少——再跑几千步它会反超。
- **UCB** 把探索花在"还不确定"的臂上，最后 100 步的平均奖励最高、累计遗憾最少。注意它前 10 步几乎是 0：它在把每个臂都试一遍。
- **累计遗憾** = 每一步"最优臂的真值 − 所选臂的真值"加起来，衡量的是**因为不知道而付出的代价**。贪心的遗憾几乎是一条直线：每一步都在稳定地吃亏。

**这也是 A/B 测试的数学模型。** 两个网页版本就是两个臂、用户点击就是奖励。传统 A/B 测试先把流量平均分配一段时间（纯探索），再全部切到赢家（纯利用）；老虎机算法边学边把流量倾向更好的版本，少把用户浪费在差版本上。自适应临床试验、"下一轮实验做哪几个条件"的自适应实验设计，用的是同一个框架。

---
## S2 · MDP 与 Q-learning

老虎机没有"局面"。真实问题里，**现在的动作会改变下一步所处的状态**：走一步棋、推一下小车、在冰面上迈一步。描述这类问题的语言叫**马尔可夫决策过程**（MDP）。今天的环境是 Gymnasium 的 **FrozenLake**：在一块 4×4 的冰面上从起点 S 走到终点 G，中间有冰洞 H。

| 概念 | 符号 | FrozenLake 里是什么 |
| --- | --- | --- |
| 状态 | $s$ | 站在哪一格（0–15，按行编号） |
| 动作 | $a$ | 左、下、右、上（0、1、2、3） |
| 转移 | $s \to s'$ | 不滑：往哪走就到哪；滑：只有 1/3 的概率走向想去的方向 |
| 奖励 | $r$ | 到达 G 得 1，其余每步 0；掉进 H 回合结束、得 0 |
| 回合 | | 从 S 出发，到 G、掉进 H 或走满 100 步为止 |

**回报**（return）是从某一步起所有未来奖励的折扣和：

$$G_t = r_{t+1} + \gamma\, r_{t+2} + \gamma^2 r_{t+3} + \cdots, \qquad 0 \le \gamma \le 1$$

折扣 $\gamma$ 表达"现在的 1 分比将来的 1 分值钱"：$\gamma = 0.95$ 时，20 步之后的奖励只值 $0.95^{20} \approx 0.36$。它也让智能体偏爱**更短**的路线。

**动作价值** $Q(s, a)$ = 在状态 $s$ 做动作 $a$、之后一直按最好的方式行动，期望能拿到的回报。它满足 **Bellman 方程**：

$$Q(s, a) = \mathbb E\Big[\, r + \gamma \max_{a'} Q(s', a') \,\Big]$$

一句话：**今天的价值 = 即时奖励 + 明天的价值（打个折）**。"明天的价值"就是走到 $s'$ 之后、挑最好的动作能拿到的 $\max_{a'} Q(s', a')$。

**Q-learning** 把这个方程变成更新规则：每走一步，就把 $Q(s, a)$ 往"即时奖励 + 打折的明天"挪一小步——

$$Q(s, a) \leftarrow Q(s, a) + \alpha\,\Big[\, \underbrace{r + \gamma \max_{a'} Q(s', a')}_{\text{目标}} - Q(s, a) \,\Big]$$

和 S1 的增量均值是同一个形状：旧值 + 步长 × (目标 − 旧值)。回合**终止**时（到 G 或掉进 H）没有明天，目标就只是 $r$。

先认识环境与 Gymnasium 1.x 的接口：`reset()` 返回 `(obs, info)`，`step(a)` 返回 5 个值 `(obs, reward, terminated, truncated, info)`。

In [ ]:
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False)
desc = env.unwrapped.desc                                # 地图：S 起点、F 冰面、H 冰洞、G 终点
for row in desc:
    print("   " + " ".join(c.decode() for c in row))
print("状态数", env.observation_space.n, "| 动作数", env.action_space.n, "（0 左、1 下、2 右、3 上）",
      "| 每回合最多", env.spec.max_episode_steps, "步")

obs, info = env.reset(seed=0)
print("\nreset() →", obs, info)
for a in [2, 2, 1, 1, 1, 2]:                             # 右、右、下、下、下、右
    obs, reward, terminated, truncated, info = env.step(a)
    print(f"step({a}) → 状态 {obs:2d} | 奖励 {reward} | terminated={terminated} | truncated={truncated}")

注意两个"结束"的区别，S4 会用到：**`terminated`** 是任务本身结束了（到了终点或掉进洞，之后没有明天）；**`truncated`** 是被时钟叫停（走满了步数上限，"明天"其实还在，只是这一局不让玩了）。

下面是表格型 Q-learning，函数结构照搬 [HF Deep RL Course Unit 2](https://huggingface.co/learn/deep-rl-course/unit2/hands-on)（Apache-2.0）的 `initialize_q_table` / `greedy_policy` / `epsilon_greedy_policy` / `train` / `evaluate_agent`。我们做了三处改动：① 所有随机性由一个 `rng` 管理（原文用 `random` 模块和 `env.action_space.sample()`），方便固定种子；② `greedy_policy` 在并列最大值之间**随机**选（原文直接 `np.argmax`，后面会看到为什么这很要紧）；③ 终止时目标里不再加"明天的价值"。

In [ ]:
def initialize_q_table(state_space, action_space):
    return np.zeros((state_space, action_space))


def greedy_policy(Qtable, state, rng):
    # 选 Q 值最大的动作；有好几个并列最大时，在它们之中均匀随机选一个
    # ✍️ 跟着老师一起写
    pass


def epsilon_greedy_policy(Qtable, state, epsilon, rng):
    # ✍️ 跟着老师一起写
    pass


def train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable,
          learning_rate, gamma, rng, policy=epsilon_greedy_policy):
    for episode in range(n_training_episodes):
        # 探索率从 max_epsilon 按指数衰减到 min_epsilon：先多探索，后多利用
        epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)
        state, info = env.reset()
        for step in range(max_steps):
            action = policy(Qtable, state, epsilon, rng)
            new_state, reward, terminated, truncated, info = env.step(action)
            # ✍️ 跟着老师一起写
            pass
            if terminated or truncated:
                break
            state = new_state
    return Qtable


def evaluate_agent(env, max_steps, n_eval_episodes, Qtable, rng, seed=10_000, policy=greedy_policy):
    # 用贪心策略（不再探索）跑 n_eval_episodes 个回合，返回每回合奖励的均值与标准差
    episode_rewards = []
    for episode in range(n_eval_episodes):
        state, info = env.reset(seed=seed + episode)
        total = 0.0
        for step in range(max_steps):
            state, reward, terminated, truncated, info = env.step(policy(Qtable, state, rng))
            total += reward
            if terminated or truncated:
                break
        episode_rewards.append(total)
    return np.mean(episode_rewards), np.std(episode_rewards)


# 全 0 的 Q 表 + 随机打破平局 = 完全随机的策略：这就是"什么都没学"的基线
rng = np.random.default_rng(0)
random_rate, _ = evaluate_agent(env, 99, 1000, initialize_q_table(16, 4), rng)
print(f"随机策略的成功率: {random_rate:.3f}")

HF Unit 2 在不滑冰面上的超参数：10000 个训练回合、学习率 0.7、$\gamma = 0.95$、每回合最多 99 步、$\varepsilon$ 从 1.0 按 $e^{-0.0005\cdot\text{回合}}$ 衰减到 0.05。

In [ ]:
def run_frozenlake(seed, slippery=False, n_episodes=10000, learning_rate=0.7, decay_rate=0.0005,
                   policy=epsilon_greedy_policy, greedy=greedy_policy, n_eval=100):
    env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=slippery)
    env.reset(seed=seed)                                 # 给环境播种（滑冰面的转移随机性）
    rng = np.random.default_rng(seed)                    # 给智能体播种（探索与平局）
    Qtable = initialize_q_table(env.observation_space.n, env.action_space.n)
    train(n_episodes, 0.05, 1.0, decay_rate, env, 99, Qtable, learning_rate, 0.95, rng, policy)
    mean_reward, _ = evaluate_agent(env, 99, n_eval, Qtable, rng, policy=greedy)
    return Qtable, mean_reward


t0 = time.time()
Q_fl, success = run_frozenlake(seed=0)
print(f"训练 10000 回合用时 {time.time() - t0:.1f} 秒 | 贪心策略成功率 {success:.2f}")
ck.frozenlake_success(success)

In [ ]:
def plot_q_table(Qtable, desc, title):
    # 每一格：底色 = max_a Q(s, a)，箭头 = 贪心动作；H 与 G 是终止格
    nrow, ncol = desc.shape
    V = Qtable.max(axis=1).reshape(nrow, ncol)
    arrows = {0: (-0.3, 0), 1: (0, 0.3), 2: (0.3, 0), 3: (0, -0.3)}      # 左、下、右、上（图的 y 轴朝下）
    fig, ax = plt.subplots(figsize=(4.4, 4.4))
    im = ax.imshow(V, cmap="Blues", vmin=0, vmax=1)
    for s in range(nrow * ncol):
        r, c = divmod(s, ncol)
        cell = desc[r, c].decode()
        if cell in "HG":
            ax.text(c, r, cell, ha="center", va="center", fontsize=16, weight="bold",
                    color="C3" if cell == "H" else "green")
            continue
        if np.all(Qtable[s] == Qtable[s, 0]):           # 四个动作一样好（没学到东西）：画一个点
            ax.plot(c, r, "k.")
        else:
            dx, dy = arrows[int(np.argmax(Qtable[s]))]
            ax.arrow(c - dx / 2, r - dy / 2, dx, dy, head_width=0.15, length_includes_head=True, color="black")
        ax.text(c, r + 0.36, f"{V[r, c]:.2f}", ha="center", va="center", fontsize=7)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, label="max_a Q(s, a)")
    ax.set_title(title, fontsize=10)
    plt.show()


print("Q 表（行 = 状态 0–15，列 = 左 / 下 / 右 / 上）：")
print(np.round(Q_fl, 3))
plot_q_table(Q_fl, desc, "Values decay by 0.95 per step away from the goal")

从终点往回看：终点旁边那格的价值是 1（下一步就拿到奖励），再往外一格是 $0.95$，再往外 $0.95^2 \approx 0.90$……**价值从终点一格一格往回传**，这正是 Bellman 方程"今天 = 即时奖励 + 打折的明天"的样子。箭头连起来就是一条最短路线。

### 一个隐蔽的坑：argmax 的平局

训练刚开始时 Q 表全是 0。`np.argmax([0, 0, 0, 0])` 永远返回 **0 = 向左**——而起点在左上角，向左就是撞墙、原地不动。如果智能体在探索率降下来之前还没碰到过终点，它的"贪心"就会变成"永远向左"。把训练预算缩到 1000 回合（探索率衰减加快 10 倍），比较两种平局规则：

In [ ]:
def greedy_first(Qtable, state, rng):
    return int(np.argmax(Qtable[state]))                 # 并列时永远返回下标最小的动作（0 = 向左）


def epsilon_greedy_first(Qtable, state, epsilon, rng):
    # 与 epsilon_greedy_policy 唯一的区别：利用时用 np.argmax
    if rng.random() > epsilon:
        return greedy_first(Qtable, state, rng)
    return int(rng.integers(Qtable.shape[1]))


for label, pol, greedy in [("np.argmax（平局选第一个）", epsilon_greedy_first, greedy_first),
                           ("随机打破平局", epsilon_greedy_policy, greedy_policy)]:
    rates = [float(run_frozenlake(seed, n_episodes=1000, decay_rate=0.005, policy=pol, greedy=greedy)[1])
             for seed in range(5)]
    print(f"{label:16s} 5 个种子的成功率: {rates}")

同样的算法、同样的预算，只改了平局规则：用 `np.argmax` 时有的种子成功率是 0——这些智能体在探索期没走到过终点（或者只走到过一次），Q 表大片仍是 0，贪心策略就一路"向左撞墙"。**平局时随机选**是表格方法里最便宜的保险。

### 把冰面变滑

`is_slippery=True` 时，你想往右走，只有 1/3 的概率真的往右，另外各 1/3 滑向两侧。同一个动作的结果变成随机的，Q 表要学的是**期望**。学习率 0.7 意味着每次更新都把估计的 70% 换成最新那一次的结果——在随机环境里，这等于被最近一次运气牵着走。比较 HF 原配的 0.7 与小一些的 0.1（各 5 个种子，每个用 1000 个评估回合）：

In [ ]:
t0 = time.time()
slippery = {lr: [run_frozenlake(seed, slippery=True, learning_rate=lr, n_eval=1000)[1] for seed in range(5)]
            for lr in (0.7, 0.1)}
print(f"滑冰面 2 个学习率 × 5 个种子，用时 {time.time() - t0:.0f} 秒")
for lr, rates in slippery.items():
    print(f"lr = {lr}: 各种子成功率 {np.round(rates, 3)} | {experiments.format_result(rates, 3)}")
ck.frozenlake_success(np.mean(slippery[0.1]), slippery=True)

两点观察：

1. 滑冰面上**最好的策略**成功率也只有约 0.73（$\gamma = 0.95$、99 步之内；可以用价值迭代精确算出来）——环境本身的随机性设了上限，0.7 左右已经接近最优。
2. lr = 0.1 的平均成功率更高，但两种学习率的种子间差异都很大（标准差约 0.15）：有的种子停在了次优策略上。RL 的结论**必须看多个种子**——这句话 S3 会让你印象更深。

---
## S3 · 策略梯度：REINFORCE

Q-learning 先学"每个动作值多少"，再挑最好的。**策略梯度**直接学策略本身：一个神经网络 $\pi_\theta(a \mid s)$，输入状态、输出每个动作的概率；训练就是调 $\theta$，让好结果出现得更频繁。状态是连续的、动作很多时，这条路比查表自然得多。

环境换成 **CartPole-v1**：小车上立着一根杆，每一步把车往左或往右推一下；杆偏离竖直超过 12° 或车出界就结束；每坚持一步得 +1，最多 500 步。状态是 4 个连续的数（车的位置、速度，杆的角度、角速度）。Gymnasium 注册表里它的"解决"标准 `reward_threshold` 是 **475**，课程统一用"**最近 100 回合平均回报 ≥ 475**"判定达标。

REINFORCE（Williams, 1992）的想法只有一句话：**一局结束后，让带来高回报的动作变得更可能。** 具体做法：

1. 用当前策略玩完整一局，记下每一步的 $(s_t, a_t, r_t)$（这里 $r_t$ 指第 $t$ 步动作之后拿到的奖励）；
2. 对每一步算 **reward-to-go**：$G_t = r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + \cdots$——只算这一步**之后**的奖励，因为过去的奖励与现在的动作无关（Spinning Up Part 3 把这一节叫作 "Don't Let the Past Distract You"）；
3. 最小化
   $$L(\theta) = -\sum_t \log \pi_\theta(a_t \mid s_t)\; G_t$$
   并做一步梯度下降：$G_t$ 越大的动作，它的对数概率被推高得越多。

它和分类的交叉熵几乎一样：把"实际做了的动作"当作标签，只是每一项乘了一个权重 $G_t$——意思不是"这个动作是对的"，而是"这个动作之后发生的事有多好"。

本节改编自 [HF Deep RL Course Unit 4](https://huggingface.co/learn/deep-rl-course/unit4/hands-on)（Apache-2.0）：策略网络 4 → 16 → 2、Adam lr 1e-2、$\gamma = 1.0$ 照搬原文。与原文有四处不同：① 原文用旧版 `gym` 的 4 元组 API，我们改成了 Gymnasium 1.x；② 原文边玩边存每一步的 `log_prob`，我们先玩完一局（不记录梯度），再把整局的状态一次性送进网络重算概率——数学上完全等价，但快得多；③ 原文每回合还把回报标准化：`(returns - returns.mean()) / (returns.std() + eps)`——它相当于减去这一局的平均回报，正是作业研究实验 5 要讨论的对象，这里先不做；④ 原文训练 1000 回合，讲义为了省时间只跑 500 回合（作业正式版用 1000）。

先看看随机乱推能坚持多久：

In [ ]:
env = gym.make("CartPole-v1")
obs, info = env.reset(seed=0)
print("初始状态（车位置、车速度、杆角度、杆角速度）:", np.round(obs, 3))
print("动作数:", env.action_space.n, "（0 向左推、1 向右推）| 最多", env.spec.max_episode_steps,
      "步 | reward_threshold =", env.spec.reward_threshold)

rng = np.random.default_rng(0)
random_returns = []
for episode in range(100):
    obs, info = env.reset(seed=episode)
    total, done = 0.0, False
    while not done:
        obs, reward, terminated, truncated, info = env.step(int(rng.integers(2)))
        total += reward
        done = terminated or truncated
    random_returns.append(total)
print(f"随机策略 100 回合的平均回报: {np.mean(random_returns):.1f}")

In [ ]:
class Policy(nn.Module):
    # HF Unit 4 的 CartPole 策略网络：4 → 16 → 2，输出向左 / 向右的概率
    def __init__(self, s_size=4, h_size=16, a_size=2):
        super().__init__()
        self.fc1 = nn.Linear(s_size, h_size)
        self.fc2 = nn.Linear(h_size, a_size)

    def forward(self, x):
        return torch.softmax(self.fc2(torch.relu(self.fc1(x))), dim=-1)


def reward_to_go(rewards, gamma):
    # G_t = r_t + γ r_{t+1} + γ² r_{t+2} + …：从最后一步倒着走，running = r_t + γ · running
    # ✍️ 跟着老师一起写
    pass


def run_episode(env, policy):
    # 用当前策略玩一局，返回这一局的状态 (T, 4)、动作 (T,)、奖励 (T,)
    obs, info = env.reset()
    states, actions, rewards = [], [], []
    done = False
    while not done:
        with torch.no_grad():                                # 收集数据时不需要梯度
            probs = policy(torch.as_tensor(obs, dtype=torch.float32))
        action = int(torch.multinomial(probs, 1))            # 按概率抽一个动作（不是取最大的！）
        states.append(obs)
        actions.append(action)
        obs, reward, terminated, truncated, info = env.step(action)
        rewards.append(reward)
        done = terminated or truncated
    return np.array(states, dtype=np.float32), np.array(actions), np.array(rewards)


print("reward_to_go([1, 1, 1, 1], γ=1)  =", reward_to_go([1.0, 1.0, 1.0, 1.0], 1.0))
print("reward_to_go([1, 0, 2], γ=0.9)   =", reward_to_go([1.0, 0.0, 2.0], 0.9))
ck.reward_to_go(reward_to_go)

训练循环：玩一局 → 算 $G_t$ → 重算整局的 $\log\pi_\theta(a_t \mid s_t)$ → 损失 → 第 8 周的"三步"（`zero_grad`、`backward`、`step`）。`probs[torch.arange(T), actions]` 从每一行里取出"实际做了的那个动作"的概率。

In [ ]:
def train_reinforce(seed, n_episodes=500, gamma=1.0, lr=1e-2):
    # 无基线的 REINFORCE：每玩一局更新一次；返回每一局的回报（= 这一局坚持了多少步）
    experiments.set_seed(seed)                     # 先固定种子再建网络：初始化与动作抽样都可复现
    env = gym.make("CartPole-v1")
    env.reset(seed=seed)                           # 只在开头给环境播一次种，之后每局的 reset() 沿这个随机序列走
    policy = Policy()
    optimizer = torch.optim.Adam(policy.parameters(), lr=lr)
    returns = []
    for episode in range(n_episodes):
        states, actions, rewards = run_episode(env, policy)
        G = reward_to_go(rewards, gamma)
        # ✍️ 跟着老师一起写
        pass
        returns.append(float(rewards.sum()))
    return returns


def episodes_to_solve(returns, threshold=475, window=100):
    # 首次"最近 window 回合平均回报 ≥ threshold"时的回合编号（从 1 数）；没达到返回 None
    for i in range(window, len(returns) + 1):
        if np.mean(returns[i - window:i]) >= threshold:
            return i
    return None


t0 = time.time()
lecture_runs = {seed: train_reinforce(seed) for seed in range(5)}
print(f"5 个种子 × 500 回合用时 {time.time() - t0:.0f} 秒")
for seed, rets in lecture_runs.items():
    solved = episodes_to_solve(rets)
    print(f"种子 {seed}: 达标回合 {solved if solved else '>500（没达标）':>12} | 最好的 100 回合平均 "
          f"{max(np.mean(rets[i - 100:i]) for i in range(100, 501)):5.1f} | 最后 100 回合平均 {np.mean(rets[-100:]):5.1f}")

In [ ]:
def moving_average(x, k=20):
    return np.convolve(x, np.ones(k) / k, mode="valid")


fig, ax = plt.subplots(figsize=(8, 4))
for seed, rets in lecture_runs.items():
    ma = moving_average(rets)
    ax.plot(np.arange(len(ma)) + 20, ma, lw=1.2, label=f"Seed {seed}")
ax.axhline(475, color="gray", ls="--", lw=1, label="Solved threshold (475)")
ax.set_xlabel("Episode")
ax.set_ylabel("Return (moving average of 20 episodes)")
ax.set_title("Same code, five seeds: REINFORCE without a baseline is a lottery")
ax.legend(fontsize=8)
plt.show()

同一份代码、同一组超参数，只换了随机种子：有的种子不到 300 回合就达标并稳住，有的始终在 100–300 之间徘徊，有的达标之后又崩回 100 以下。**如果你只跑一个种子，你对这个算法的结论完全取决于运气。** 这就是本课程要求 RL 实验至少跑 5 个种子的原因。

### 为什么这么不稳？——基线

看损失里的权重 $G_t$：CartPole 每一步奖励都是 +1，所以**每一个**做过的动作，$G_t$ 都是正的——不论好坏全部被"鼓励"，只是程度不同。好与坏的信号藏在 $G_t$ 的**相对**大小里，而它的绝对大小又随这一局长短剧烈变化（一局 20 步和一局 300 步）。用一局数据估计的梯度因此方差极大。

办法是减去一个**基线** $b_t$：

$$L(\theta) = -\sum_t \log \pi_\theta(a_t \mid s_t)\,\big(G_t - b_t\big)$$

只要 $b_t$ **不依赖于这一步实际做的动作**——例如只依赖状态，或者只用过去回合的数据算出来——减掉它就**不改变梯度的期望**（Spinning Up Part 3 "Baselines in Policy Gradients" 有两行证明），却能大大降低方差：比"平常"好的动作被鼓励、比"平常"差的被抑制。$G_t - b_t$ 叫作**优势**（advantage）。

基线怎么选，是本周作业"研究实验 5"的主题：A 无基线；B 跨回合、按时间步记录的滑动平均基线；C 在 B 的基础上再把优势标准化。**先剧透一句：C 不一定比 B 好**——作业要你用 5 个种子的数据说话，并解释为什么。

---
## S4 · 奖励设计与奖励欺骗

强化学习只做一件事：**最大化你写下的奖励**。奖励写错了，它会非常努力地去做错的事。

经典的例子是 OpenAI 的博客 [Faulty Reward Functions in the Wild](https://openai.com/index/faulty-reward-functions/)（Jack Clark & Dario Amodei, 2016）：在赛艇游戏 CoastRunners 里，奖励是游戏得分，而撞到赛道上的得分目标就会加分。智能体学会了在一个小潟湖里不停兜圈、反复撞同一批目标——着火、逆行、从不完赛，得分却比好好跑完全程还高。

我们用 **MountainCar-v0** 亲手复现这种现象。一辆动力不足的小车停在山谷里，目标是开到右边山顶的旗子（位置 $x \ge 0.5$）。发动机太弱，不能直接开上去：必须先往左冲上坡，再借势往右——**先远离目标，才能到达目标**。每回合最多 200 步，动作是向左推、不推、向右推。

同一个表格型 Q-learning（S2 的 `epsilon_greedy_policy` 和更新规则原样搬来；连续的位置与速度各切成 20 格，变成 400 个离散状态），只换奖励：

| 奖励 | 定义 | 设计者的想法 |
| --- | --- | --- |
| ① 稀疏 | 到达旗子 +1，其余每步 0 | 只奖励真正想要的结果 |
| ② 势函数塑形 | ① + $\gamma\,\Phi(s') - \Phi(s)$，$\Phi$ = 机械能（动能 + 重力势能） | 每一步"攒到能量"都给即时反馈 |
| ③ 速度奖励 | ① + 每步 $10\,\lvert v' \rvert$ | "跑得快就离成功近"，听起来也合理…… |

② 的形式来自 Ng, Harada & Russell, *Policy invariance under reward transformations*（ICML 1999）：如果加上的奖励是一个"势能差" $\gamma\Phi(s') - \Phi(s)$，那么沿任何一条轨迹它们加起来都会几乎全部相消（像爬山的高度差只取决于起点和终点），**最优策略因此不变**——它只改变学得快慢，不改变要学的东西。③ 不是势能差，没有这个保证。

（为什么不直接用环境自带的奖励"每步 −1"？因为它配上全 0 的 Q 表，等于给所有没去过的状态打了高分——它们的真实价值都是负的——智能体会被驱动去探索新状态，反而能学会。这叫**乐观初始化**。要演示"稀疏奖励学不会"，必须用"只在终点 +1"。）

In [ ]:
GAMMA_MC = 0.99


def discretize(obs, n_bins=20):
    # (位置 x ∈ [-1.2, 0.6], 速度 v ∈ [-0.07, 0.07]) → 0 … n_bins² − 1 的整数状态
    x, v = obs
    i = min(int((x + 1.2) / 1.8 * n_bins), n_bins - 1)
    j = min(int((v + 0.07) / 0.14 * n_bins), n_bins - 1)
    return i * n_bins + j


def potential(obs):
    # 机械能：动能 ½v² + 重力势能。MountainCar 的重力项是 0.0025·cos(3x)，对应的"高度"∝ sin(3x)/3；×1000 只是换个单位
    x, v = obs
    return 1000.0 * (0.5 * v ** 2 + 0.0025 * np.sin(3 * x) / 3)


def reward_sparse(obs, obs_next, terminated):
    # ✍️ 跟着老师一起写
    pass


def reward_potential(obs, obs_next, terminated):
    # ✍️ 跟着老师一起写
    pass


def reward_velocity(obs, obs_next, terminated):
    # ✍️ 跟着老师一起写
    pass


def train_mountaincar(reward_fn, seed, n_episodes=500, n_bins=20, alpha=0.5, epsilon=0.1, bootstrap_truncated=True):
    # 表格 Q-learning；返回 Q 表与"每个训练回合是否到达了旗子"
    # bootstrap_truncated=False 只用于后面的对照实验：把"走满 200 步被截断"也当成终止
    rng = np.random.default_rng(seed)
    env = gym.make("MountainCar-v0")
    Q = initialize_q_table(n_bins * n_bins, env.action_space.n)
    reached = []
    obs, info = env.reset(seed=seed)
    for episode in range(n_episodes):
        if episode > 0:
            obs, info = env.reset()
        s = discretize(obs, n_bins)
        while True:
            a = epsilon_greedy_policy(Q, s, epsilon, rng)
            obs_next, _, terminated, truncated, info = env.step(a)      # 丢掉环境自带的奖励（每步 −1）
            r = reward_fn(obs, obs_next, terminated)
            s_next = discretize(obs_next, n_bins)
            no_tomorrow = terminated or (truncated and not bootstrap_truncated)
            target = r if no_tomorrow else r + GAMMA_MC * Q[s_next].max()  # 默认：截断（走满 200 步）不是终止，照样加"明天"
            Q[s, a] += alpha * (target - Q[s, a])
            obs, s = obs_next, s_next
            if terminated or truncated:
                break
        reached.append(terminated)
    return Q, np.array(reached)


# 训练之前先单独检查塑形函数本身：只看训练后的到达率，抓不到"漏了 γ""终止时没把 Φ 置 0"这类错误
ck.potential_shaping(reward_potential, potential, GAMMA_MC)

In [ ]:
reward_designs = {"Sparse (+1 at flag)": reward_sparse,
                  "Potential shaping": reward_potential,
                  "Velocity bonus": reward_velocity}
MC_SEEDS = (0, 1, 2)
t0 = time.time()
mc = {name: [train_mountaincar(fn, seed) for seed in MC_SEEDS] for name, fn in reward_designs.items()}
print(f"3 种奖励 × 3 个种子 × 500 回合，用时 {time.time() - t0:.0f} 秒")
reach_rate = {}
for name, runs in mc.items():
    firsts = [int(np.argmax(reached)) + 1 if reached.any() else None for _, reached in runs]
    rates = [float(reached.mean()) for _, reached in runs]
    reach_rate[name] = float(np.mean(rates))
    print(f"{name:20s} 首次到达旗子的回合 {firsts} | 训练回合中到达旗子的比例 {np.round(rates, 3)}"
          f"（平均 {reach_rate[name]:.3f}）")
ck.reward_hacking_demo(*reach_rate.values())

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))
for (name, runs), color in zip(mc.items(), ["C7", "C0", "C3"]):
    style = dict(lw=5, alpha=0.4) if name.startswith("Sparse") else \
        dict(lw=1.4, ls="--") if name.startswith("Velocity") else dict(lw=1.4)
    for i, (_, reached) in enumerate(runs):
        ax.plot(np.arange(1, len(reached) + 1), np.cumsum(reached), color=color,
                label=name if i == 0 else None, **style)
ax.set_xlabel("Training episode")
ax.set_ylabel("Episodes that reached the flag (cumulative)")
ax.set_title("Only potential shaping ever reaches the flag (3 seeds per reward)")
ax.legend()
plt.show()

稀疏奖励与速度奖励的线（灰、红）贴在 0 上——一次都没到过旗子；势函数塑形（蓝）在三个种子上都学会了上山，但快慢差别很大。速度奖励的智能体在干什么？训练完之后，用纯贪心策略（不再探索）各跑一局看看：

In [ ]:
def greedy_rollout(Q, reward_fn, seed=123, n_bins=20):
    # 纯贪心跑一局；返回每一步的位置、速度，这一局按 reward_fn 计的总奖励，以及是否到达旗子
    env = gym.make("MountainCar-v0")
    rng = np.random.default_rng(seed)
    obs, info = env.reset(seed=seed)
    xs, vs, total = [obs[0]], [obs[1]], 0.0
    while True:
        a = greedy_policy(Q, discretize(obs, n_bins), rng)
        obs_next, _, terminated, truncated, info = env.step(a)
        total += reward_fn(obs, obs_next, terminated)
        obs = obs_next
        xs.append(obs[0])
        vs.append(obs[1])
        if terminated or truncated:
            break
    return np.array(xs), np.array(vs), total, terminated


fig, ax = plt.subplots(figsize=(9.5, 3.8))
for (name, runs), color in zip(mc.items(), ["C7", "C0", "C3"]):
    Q_seed0 = runs[0][0]                                  # 种子 0 训练出的 Q 表
    xs, vs, total, reached = greedy_rollout(Q_seed0, reward_designs[name])
    ax.plot(xs, color=color, lw=1.5, label=f"{name} (reached flag: {reached})")
    print(f"{name:20s} 到达旗子 {str(reached):5s} | 用了 {len(xs) - 1:3d} 步 | 最远到 x = {xs.max():5.2f} | "
          f"平均 |v| = {np.abs(vs).mean():.4f} | 按它自己的奖励这一局得 {total:6.2f} 分")
ax.axhline(0.5, color="green", ls="--", lw=1, label="Flag (x = 0.5)")
ax.axhline(-np.pi / 6, color="gray", ls=":", lw=1, label="Valley bottom")
ax.set_xlabel("Step")
ax.set_ylabel("Car position x")
ax.set_title("The velocity-bonus agent swings in the valley instead of climbing out")
ax.legend(fontsize=8, loc="center left", bbox_to_anchor=(1.01, 0.5))
plt.tight_layout()
plt.show()

三条线讲了三件事：

1. **稀疏奖励下没有可学的东西。** 随机乱推从来碰不到旗子，所有奖励都是 0，Q 表始终全 0——训练 500 回合等于没训练，灰线就是一个随机策略在谷底附近晃。
2. **势函数塑形把信号变密了。** 每一步"攒到能量"立刻有正反馈，智能体很快学会"先往左冲、再往右上"——但三个种子的快慢差别很大：塑形不是魔法，它只是让学习有了方向。
3. **速度奖励被"钻了空子"。** 这个智能体确实学会了猛烈地来回摆（平均速度明显高于随机乱推），却**从不去碰旗子**。为什么？碰到旗子回合就结束、速度奖励也就停了；而一直摆下去，每一步都有奖励——$\gamma = 0.99$ 时这条奖励流的折扣和有十几分（每步最多 0.7 分，无穷和的上限是 $0.7/(1-\gamma) = 70$，它实际摆得没那么快），远大于旗子的 +1。智能体的状态里只有位置和速度，没有"还剩几步"，所以在它看来，每一步都还可以一直摆下去。上面的输出也印证了这一点：按各自的奖励算，摆来摆去的那一局得分比真正上山的那一局高得多。**它最大化的正是我们写下的奖励**——这就是奖励欺骗（reward hacking），和 CoastRunners 里兜圈刷分的小船是同一回事。

**对照实验：是"截断照样自举"惹的祸吗？** S2 刚强调过：走满 200 步被截断不是终止，目标里照样加"明天"。你也许会猜，正是这一点让 Q-learning 以为能"永远摆下去"。检验它只需要改一处——把截断也当成终止（截断那一步的目标只取 $r$），其余完全不变，同样 3 个种子：

In [ ]:
# 对照：bootstrap_truncated=False 把"走满 200 步被截断"也当成终止，其余与上面完全相同
t0 = time.time()
mc_control = [train_mountaincar(reward_velocity, seed, bootstrap_truncated=False) for seed in MC_SEEDS]
s_start = discretize((-0.5, 0.0))                     # 起点附近（谷底、静止）所在的格子
for label, runs in [("截断照样自举（上面的结果）", mc["Velocity bonus"]), ("对照：截断当作终止", mc_control)]:
    reached = [int(r.sum()) for _, r in runs]
    v_start = [float(Q[s_start].max()) for Q, _ in runs]
    print(f"{label:14s} 训练中到达旗子的回合数 {reached} | 起点格子的 max_a Q {np.round(v_start, 1)}")
print(f"对照实验用时 {time.time() - t0:.0f} 秒")

两种写法都**一次也没有碰到旗子**，起点的价值也只差一点（都在 10–13 分左右）。原因是：状态里没有"剩余时间"，截断那一步"没有明天"的目标只落在 200 步里随机的某一格上，对其余 199 次自举几乎没有影响——智能体学到的仍然是"从这里一直摆下去"值多少分。所以**截断照样自举是对的（S2），但它不是这次奖励欺骗的原因**，把截断改成终止也治不好它；要治，得改奖励本身。

两条经验：① 想加"中间奖励"，尽量写成势能差的形式（不改变最优策略）；② 加了任何塑形之后，一定用**原本的目标**（这里是"到没到旗子"）去评估，而不是看训练奖励涨没涨。

### 从 REINFORCE 到 PPO 与 RLHF（概念）

今天的 REINFORCE 是策略梯度最朴素的形式。现在最常用的 **PPO**（Schulman et al., *Proximal Policy Optimization Algorithms*, 2017，[arXiv:1707.06347](https://arxiv.org/abs/1707.06347)）在它的基础上做了三件事：用一个价值网络 $V(s)$ 估计基线（即 actor-critic）；同一批数据做多轮小批量更新、不浪费；把新旧策略的概率比 $\pi_\theta / \pi_{\text{old}}$ 裁剪在 $[1-\epsilon, 1+\epsilon]$ 之内，防止一步走得太远把策略改坏。

**RLHF**（reinforcement learning from human feedback）连"奖励"本身也从数据里学：Christiano et al., *Deep reinforcement learning from human preferences*（2017，[arXiv:1706.03741](https://arxiv.org/abs/1706.03741)）让人比较两段行为哪个更好，用这些偏好训练一个奖励模型，再用 RL 去优化它；Ouyang et al.（InstructGPT，2022，[arXiv:2203.02155](https://arxiv.org/abs/2203.02155)）把这套流程用在语言模型上——监督微调 → 训练奖励模型 → 用 PPO 优化——此后成为对话模型对齐训练的常用流程。

| | 本周的 CartPole REINFORCE | PPO | RLHF（InstructGPT 式） |
| --- | --- | --- | --- |
| 策略 | 4 → 16 → 2 的小网络 | 任意策略网络 | 预训练语言模型 |
| 一个"动作" | 向左 / 向右推 | 环境的动作 | 生成下一个 token |
| 奖励从哪来 | 环境：每坚持一步 +1 | 环境 | **奖励模型**（由人类的两两偏好训练） |
| 方差缩减 | 无（作业里加基线） | 价值网络 $V(s)$ 估计优势 | 同 PPO |
| 一次更新 | 每局一步梯度 | 同一批数据多轮小批量 + 裁剪概率比 | PPO + 惩罚偏离原模型太远（KL 项） |
| 奖励欺骗的风险 | 低：奖励就是目标本身 | 取决于奖励怎么写 | 高：模型可能学会讨好奖励模型，而不是真的变好 |

最后一行把 S4 串了起来：RLHF 里的奖励模型也是一个"写下来的奖励"，它和"人真正想要的"之间的缝隙，就是 CoastRunners 那条小船钻的空子。

---
## Capstone：主结果

本周的 Capstone 里程碑是**主结果**：用第 12 周设计文档里定下的主实验配置，跑出**至少 3 个种子**（RL 类选题建议 5 个）的结果，报告均值 ± 标准差，画出带误差棒（或均值 ± 标准差带）的主图，并和第 16 周的基线放进同一张表。写进实验记录本（`templates/lab_notebook.md` 的格式，作业 E 部分有模板）。

今天的两件事可以直接用上：S3 的五条曲线提醒你，**一个种子的结果什么也说明不了**；S4 提醒你，**评估要用你真正关心的指标**，而不是训练时优化的那个代理量。下周（第 19 周）做消融与报告初稿，所以这周的主结果要能一键复现：固定种子、记下配置、把结果存成文件。

### 本周小结

- 强化学习：智能体自己收集数据；核心难题是**探索与利用**和**信用分配**
- 多臂老虎机：增量均值 $Q \leftarrow Q + \frac1n(R - Q)$；ε-greedy 盲目探索，UCB 向"不确定"的臂探索；用平均奖励、最优动作比例、累计遗憾比较；它也是 A/B 测试与自适应实验设计的模型
- MDP：状态、动作、奖励、回报 $G_t$、折扣 $\gamma$；Bellman 方程 = 今天的价值 = 即时奖励 + 打折的明天
- Q-learning：$Q(s,a) \leftarrow Q(s,a) + \alpha[r + \gamma\max_{a'}Q(s',a') - Q(s,a)]$，终止时没有明天、截断时照样自举；argmax 平局要随机打破；随机环境里学习率要小
- REINFORCE：玩完一局 → reward-to-go → 损失 $-\sum_t \log\pi(a_t\mid s_t)\,G_t$；种子间方差巨大，所以至少 5 个种子；减去不依赖动作的基线不改变期望、降低方差
- 奖励设计：稀疏奖励学不会；势函数塑形不改变最优策略、让学习变快；随手加的中间奖励会被钻空子（奖励欺骗）；评估永远看真正的目标
- PPO = 策略梯度 + 价值基线 + 裁剪的多轮更新；RLHF = 从人类偏好学奖励模型 + PPO